In [0]:
%sql

USE CATALOG gitobservatory;

**REPOSITORY METADATA**

In [0]:
from datetime import datetime

config = spark.table("gitobservatory.bronze.pipeline_config").first()

analysis_window = config["analysis_window"]

analysis_since = datetime.fromisoformat(
    config["analysis_since"]
)

In [0]:
from pyspark.sql.types import *
df_repo_metadata=spark.createDataFrame(
    data=[],
    schema=StructType(
        [
            StructField('github_repo_id', LongType(), True),
            StructField('owner', StringType(), True),
            StructField('repository_name', StringType(), True),
            StructField('repo_full_name', StringType(), True),
            StructField('source_system', StringType(), True),
            StructField('ingestion_timestamp', TimestampType(), True),
            StructField("repo_node_id", StringType(), True),
            StructField("raw_json", StringType(), True)
        ]
    )
)

df_repo_metadata.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.bronze.repository_metadata")


In [0]:
from dotenv import load_dotenv
import os
load_dotenv()
GITHUB_TOKEN=os.getenv("GITHUB_TOKEN")


In [0]:
headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp


config = spark.table("gitobservatory.bronze.repository_config")

for repo in config.collect():
    owner = repo["owner"]
    repository = repo["repository_name"]
    url = f"https://api.github.com/repos/{owner}/{repository}"
    print(f"Processing {owner}/{repository}")
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        print(f"Success in pulling metadata for {owner}/{repository}")
        raw_json = json.dumps(data)
        record = {
            "github_repo_id": data["id"],
            "owner": data.get("owner", {}).get("login", ""),
            "repository_name": data["name"],
            "repo_full_name": data["full_name"],
            "source_system": "github",
            "repo_node_id":data["node_id"],
            "raw_json": raw_json
        }
        df = spark.createDataFrame([record])
        df = df.withColumn("ingestion_timestamp",current_timestamp())

        df.write.format("delta").mode("append").saveAsTable("gitobservatory.bronze.repository_metadata")
    else:
        print(f"Failed for {owner}/{repository}")

Processing dbt-labs/dbt-core
Success in pulling metadata for dbt-labs/dbt-core
Processing PrefectHQ/prefect
Success in pulling metadata for PrefectHQ/prefect


**PRs INGESTION**

In [0]:
from datetime import datetime, timedelta
import requests
import json

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    BooleanType,
    TimestampType
)


print(f"Analyzing Pull Requests since {analysis_since.strftime('%Y-%m-%d')}")
spark.sql("TRUNCATE TABLE gitobservatory.bronze.pull_requests")
config = spark.table("gitobservatory.bronze.repository_config")

records = []

for repo in config.collect():
    OWNER = repo["owner"]
    REPOSITORY = repo["repository_name"]
    print(f"\nProcessing {OWNER}/{REPOSITORY}")
    page = 1
    while True:
        url = (
            f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/pulls"
            f"?state=all&per_page=100&page={page}"
        )
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed for {OWNER}/{REPOSITORY} with status code {response.status_code}")
            break
        data = response.json()
        if len(data) == 0:
            print(f"Finished fetching pull requests for {OWNER}/{REPOSITORY}")
            break
        print(f"Fetched Page {page}: {len(data)} pull requests")
        stop_fetching = False
        for pr in data:
            created_at = datetime.strptime(
                pr["created_at"],
                "%Y-%m-%dT%H:%M:%SZ"
            )
            if created_at < analysis_since:
                stop_fetching = True
                break

            record = {
                "pr_id": pr.get("id"),
                "pr_node_id": pr.get("node_id"),
                "pr_number": pr.get("number"),
                "repo_id": pr.get("base", {}).get("repo", {}).get("id"),
                "repo_node_id": pr.get("base", {}).get("repo", {}).get("node_id"),
                "repo_full_name": pr.get("base", {}).get("repo", {}).get("full_name"),
                "user_id": pr.get("user", {}).get("id"),
                "user_login": pr.get("user", {}).get("login"),
                "author_association": pr.get("author_association"),
                "state": pr.get("state"),
                "draft": pr.get("draft"),
                "html_url": pr.get("html_url"),
                "created_at": pr.get("created_at"),
                "updated_at": pr.get("updated_at"),
                "closed_at": pr.get("closed_at"),
                "merged_at": pr.get("merged_at"),
                "source_system": "github",
                "raw_json": json.dumps(pr)
            }
            records.append(record)
        if stop_fetching:
            break
        page += 1

schema = StructType([
    StructField("pr_id", LongType(), True),
    StructField("pr_node_id", StringType(), True),
    StructField("pr_number", LongType(), True),
    StructField("repo_id", LongType(), True),
    StructField("repo_node_id", StringType(), True),
    StructField("repo_full_name", StringType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_login", StringType(), True),
    StructField("author_association", StringType(), True),
    StructField("state", StringType(), True),
    StructField("draft", BooleanType(), True),
    StructField("html_url", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("closed_at", StringType(), True),
    StructField("merged_at", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)
])
df = spark.createDataFrame(records, schema=schema)

df = df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("gitobservatory.bronze.pull_requests")
print(f"\nSuccessfully ingested {len(records)} pull requests.")

/home/spark-233766c2-48cf-4c12-8882-15/.ipykernel/306/command-7585012712732726-131903755:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  analysis_since = datetime.utcnow() - timedelta(days=365 * ANALYSIS_YEARS)


Analyzing Pull Requests since 2024-07-01

Processing dbt-labs/dbt-core
Fetched Page 1: 100 pull requests
Fetched Page 2: 100 pull requests
Fetched Page 3: 100 pull requests
Fetched Page 4: 100 pull requests
Fetched Page 5: 100 pull requests
Fetched Page 6: 100 pull requests
Fetched Page 7: 100 pull requests
Fetched Page 8: 100 pull requests
Fetched Page 9: 100 pull requests
Fetched Page 10: 100 pull requests
Fetched Page 11: 100 pull requests
Fetched Page 12: 100 pull requests
Fetched Page 13: 100 pull requests
Fetched Page 14: 100 pull requests
Fetched Page 15: 100 pull requests
Fetched Page 16: 100 pull requests
Reached pull requests older than 2 years for dbt-labs/dbt-core

Processing PrefectHQ/prefect
Fetched Page 1: 100 pull requests
Fetched Page 2: 100 pull requests
Fetched Page 3: 100 pull requests
Fetched Page 4: 100 pull requests
Fetched Page 5: 100 pull requests
Fetched Page 6: 100 pull requests
Fetched Page 7: 100 pull requests
Fetched Page 8: 100 pull requests
Fetched Page 

**ISSUES BRONZE SCHEMA**

In [0]:
from datetime import datetime, timedelta
import requests
import json

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    IntegerType,
    TimestampType
)

# =====================================================
# Configuration
# =====================================================

print(f"Analyzing Issues since {analysis_since.strftime('%Y-%m-%d')}")

# =====================================================
# Clear Bronze Table
# =====================================================

spark.sql("TRUNCATE TABLE gitobservatory.bronze.issues")

# =====================================================
# Read Repository Configuration
# =====================================================

config = spark.table("gitobservatory.bronze.repository_config")
repo_metadata = spark.table("gitobservatory.bronze.repository_metadata")
repo_lookup = {}
for row in repo_metadata.collect():

    repo_lookup[row["repo_full_name"]] = {
        "repo_id": row["github_repo_id"],
        "repo_node_id": row["repo_node_id"]
    }
records = []
for repo in config.collect():
    OWNER = repo["owner"]
    REPOSITORY = repo["repository_name"]
    repo_full_name = f"{OWNER}/{REPOSITORY}"
    print(f"\nProcessing {repo_full_name}")
    page = 1
    while True:
        url = (
            f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/issues"
            f"?state=all&per_page=100&page={page}"
        )
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed for {repo_full_name} with status code {response.status_code}")
            break
        data = response.json()
        if len(data) == 0:
            print(f"Finished fetching issues for {repo_full_name}")
            break
        print(f"Fetched Page {page}: {len(data)} records")
        stop_fetching = False
        for issue in data:
            if "pull_request" in issue:
                continue

            created_at = datetime.strptime(issue["created_at"],"%Y-%m-%dT%H:%M:%SZ")

            if created_at < analysis_since:
                stop_fetching = True
                break

            repo_info = repo_lookup.get(repo_full_name, {})
            record = {
                "issue_id": issue.get("id"),
                "issue_node_id": issue.get("node_id"),
                "issue_number": issue.get("number"),
                "repo_id": repo_info.get("repo_id"),
                "repo_node_id": repo_info.get("repo_node_id"),
                "repo_full_name": repo_full_name,
                "user_id": issue.get("user", {}).get("id"),
                "user_login": issue.get("user", {}).get("login"),
                "author_association": issue.get("author_association"),
                "state": issue.get("state"),
                "state_reason": issue.get("state_reason"),
                "comments": issue.get("comments"),
                "html_url": issue.get("html_url"),
                "created_at": issue.get("created_at"),
                "updated_at": issue.get("updated_at"),
                "closed_at": issue.get("closed_at"),
                "source_system": "github",
                "raw_json": json.dumps(issue)
            }
            records.append(record)
        if stop_fetching:
            break
        page += 1


schema = StructType([
    StructField("issue_id", LongType(), True),
    StructField("issue_node_id", StringType(), True),
    StructField("issue_number", LongType(), True),
    StructField("repo_id", LongType(), True),
    StructField("repo_node_id", StringType(), True),
    StructField("repo_full_name", StringType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_login", StringType(), True),
    StructField("author_association", StringType(), True),
    StructField("state", StringType(), True),
    StructField("state_reason", StringType(), True),
    StructField("comments", IntegerType(), True),
    StructField("html_url", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("closed_at", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)

])

df = spark.createDataFrame(records, schema=schema)
df = df.withColumn("ingestion_timestamp",current_timestamp()
)

df.write.format("delta").mode("append").saveAsTable("gitobservatory.bronze.issues")
print(f"\nSuccessfully ingested {len(records)} issues.")


/home/spark-233766c2-48cf-4c12-8882-15/.ipykernel/306/command-7585012712732717-3227608179:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  analysis_since = datetime.utcnow() - timedelta(days=365 * ANALYSIS_YEARS)


Analyzing Issues since 2024-07-01

Processing dbt-labs/dbt-core
Fetched Page 1: 100 records
Fetched Page 2: 100 records
Fetched Page 3: 100 records
Fetched Page 4: 100 records
Fetched Page 5: 100 records
Fetched Page 6: 100 records
Fetched Page 7: 100 records
Fetched Page 8: 100 records
Fetched Page 9: 100 records
Fetched Page 10: 100 records
Fetched Page 11: 100 records
Fetched Page 12: 100 records
Fetched Page 13: 100 records
Fetched Page 14: 100 records
Fetched Page 15: 100 records
Fetched Page 16: 100 records
Fetched Page 17: 100 records
Fetched Page 18: 100 records
Fetched Page 19: 100 records
Fetched Page 20: 100 records
Fetched Page 21: 100 records
Fetched Page 22: 100 records
Fetched Page 23: 100 records
Fetched Page 24: 100 records
Fetched Page 25: 100 records
Fetched Page 26: 100 records
Fetched Page 27: 100 records
Fetched Page 28: 100 records
Fetched Page 29: 100 records
Fetched Page 30: 100 records
Fetched Page 31: 100 records
Fetched Page 32: 100 records
Fetched Page 33: 

**CONTRIBUTIONS BRONZE TABLE**

In [0]:
import requests
import json

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)


spark.sql("TRUNCATE TABLE gitobservatory.bronze.contributors")

config = spark.table("gitobservatory.bronze.repository_config")
repo_metadata = spark.table("gitobservatory.bronze.repository_metadata")

repo_lookup = {}

for row in repo_metadata.collect():

    repo_lookup[row["repo_full_name"]] = {
        "repo_id": row["github_repo_id"],      
        "repo_node_id": row["repo_node_id"]
    }

records = []

for repo in config.collect():

    OWNER = repo["owner"]
    REPOSITORY = repo["repository_name"]

    REPO_FULL_NAME = f"{OWNER}/{REPOSITORY}"

    repo_info = repo_lookup.get(REPO_FULL_NAME)

    if repo_info is None:
        print(f"Repository metadata not found for {REPO_FULL_NAME}")
        continue

    REPO_ID = repo_info["repo_id"]
    REPO_NODE_ID = repo_info["repo_node_id"]

    print(f"\nProcessing {REPO_FULL_NAME}")

    page = 1
    while True:
        url = (
            f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/contributors"
            f"?per_page=100&page={page}"
        )
        response = requests.get(url, headers=headers)
        if response.status_code == 422:
            print(f"Finished fetching contributors for {REPO_FULL_NAME}")
            break

        if response.status_code != 200:
            print(f"Failed for {REPO_FULL_NAME} with status code {response.status_code}")
            break

        data = response.json()

        if len(data) == 0:
            print(f"Finished fetching contributors for {REPO_FULL_NAME}")
            break

        print(f"Fetched Page {page}: {len(data)} contributors")

        for contributor in data:
            record = {
                "contributor_id": contributor.get("id"),
                "contributor_node_id": contributor.get("node_id"),
                "repo_id": REPO_ID,
                "repo_node_id": REPO_NODE_ID,
                "repo_full_name": REPO_FULL_NAME,
                "user_login": contributor.get("login"),
                "user_type": contributor.get("type"),
                "site_admin": contributor.get("site_admin"),
                "contributions": contributor.get("contributions"),
                "html_url": contributor.get("html_url"),
                "source_system": "github",
                "raw_json": json.dumps(contributor)
            }
            records.append(record)
        page += 1

schema = StructType([
    StructField("contributor_id", LongType(), True),
    StructField("contributor_node_id", StringType(), True),
    StructField("repo_id", LongType(), True),
    StructField("repo_node_id", StringType(), True),
    StructField("repo_full_name", StringType(), True),
    StructField("user_login", StringType(), True),
    StructField("user_type", StringType(), True),
    StructField("site_admin", BooleanType(), True),
    StructField("contributions", IntegerType(), True),
    StructField("html_url", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)
])

df = spark.createDataFrame(records, schema=schema)
df = df.withColumn("ingestion_timestamp",current_timestamp())

df.write.format("delta").mode("append").saveAsTable("gitobservatory.bronze.contributors")
print(f"\nSuccessfully ingested {len(records)} contributors.")



Processing dbt-labs/dbt-core
Fetched Page 1: 100 contributors
Fetched Page 2: 100 contributors
Fetched Page 3: 100 contributors
Fetched Page 4: 76 contributors
Finished fetching contributors for dbt-labs/dbt-core

Processing PrefectHQ/prefect
Fetched Page 1: 100 contributors
Fetched Page 2: 100 contributors
Fetched Page 3: 100 contributors
Fetched Page 4: 100 contributors
Fetched Page 5: 23 contributors
Finished fetching contributors for PrefectHQ/prefect

Successfully ingested 799 contributors.


**PULL REQUEST REVIEWS**

In [0]:
from pyspark.sql.types import *
df = spark.createDataFrame(
    [],
    StructType([
        StructField("review_id", LongType(), True),
        StructField("pr_id", LongType(), True),
        StructField("reviewer_id", LongType(), True),
        StructField("reviewer_login", StringType(), True),
        StructField("state", StringType(), True),
        StructField("submitted_at", StringType(), True),
        StructField("body", StringType(), True),
        StructField("html_url", StringType(), True),
        StructField("source_system", StringType(), True),
        StructField("raw_json", StringType(), True),
        StructField("ingestion_timestamp", TimestampType(), True)
    ])
)

df.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.bronze.pull_request_reviews")

In [0]:
from datetime import datetime, timedelta
import requests
import json

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *

print(f"Analyzing PR Reviews since {analysis_since.strftime('%Y-%m-%d')}")
spark.sql("TRUNCATE TABLE gitobservatory.bronze.pull_request_reviews")
pr_df = spark.table("gitobservatory.bronze.pull_requests")
records = []

for pr in pr_df.collect():
    owner = pr["repo_full_name"].split("/")[0]
    repository = pr["repo_full_name"].split("/")[1]
    pr_number = pr["pr_number"]
    created_at = datetime.strptime(
        pr["created_at"],
        "%Y-%m-%dT%H:%M:%SZ"
    )
    if created_at < analysis_since:
        continue
    print(f"Processing {owner}/{repository} PR #{pr_number}")
    url = f"https://api.github.com/repos/{owner}/{repository}/pulls/{pr_number}/reviews"
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed PR {pr_number}")
        continue
    reviews = response.json()
    for review in reviews:
        record = {
            "review_id": review.get("id"),
            "pr_id": pr["pr_id"],
            "reviewer_id": review.get("user", {}).get("id"),
            "reviewer_login": review.get("user", {}).get("login"),
            "state": review.get("state"),
            "submitted_at": review.get("submitted_at"),
            "body": review.get("body"),
            "html_url": review.get("html_url"),
            "source_system": "github",
            "raw_json": json.dumps(review)
        }
        records.append(record)

schema = StructType([
    StructField("review_id", LongType(), True),
    StructField("pr_id", LongType(), True),
    StructField("reviewer_id", LongType(), True),
    StructField("reviewer_login", StringType(), True),
    StructField("state", StringType(), True),
    StructField("submitted_at", StringType(), True),
    StructField("body", StringType(), True),
    StructField("html_url", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)
])

df = spark.createDataFrame(records, schema=schema)
df = df.withColumn("ingestion_timestamp",current_timestamp())

df.write.format("delta").mode("append").saveAsTable("gitobservatory.bronze.pull_request_reviews")

print(f"Successfully ingested {len(records)} reviews.")



/home/spark-233766c2-48cf-4c12-8882-15/.ipykernel/306/command-7585012712732741-4089621846:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  analysis_since = datetime.utcnow() - timedelta(days=365 * ANALYSIS_YEARS)


Analyzing PR Reviews since 2025-12-31
Processing PrefectHQ/prefect PR #22046
Processing PrefectHQ/prefect PR #22045
Processing PrefectHQ/prefect PR #22044
Processing PrefectHQ/prefect PR #22043
Processing PrefectHQ/prefect PR #22041
Processing PrefectHQ/prefect PR #22040
Processing PrefectHQ/prefect PR #22039
Processing PrefectHQ/prefect PR #22038
Processing PrefectHQ/prefect PR #22036
Processing PrefectHQ/prefect PR #22035
Processing PrefectHQ/prefect PR #22034
Processing PrefectHQ/prefect PR #22033
Processing PrefectHQ/prefect PR #22032
Processing PrefectHQ/prefect PR #22031
Processing PrefectHQ/prefect PR #22030
Processing PrefectHQ/prefect PR #22029
Processing PrefectHQ/prefect PR #22026
Processing PrefectHQ/prefect PR #22025
Processing PrefectHQ/prefect PR #22024
Processing PrefectHQ/prefect PR #22023
Processing PrefectHQ/prefect PR #22022
Processing PrefectHQ/prefect PR #22021
Processing PrefectHQ/prefect PR #22020
Processing PrefectHQ/prefect PR #22019
Processing PrefectHQ/prefe

**WORKFLOW RUNS**

In [0]:
from pyspark.sql.types import *

df = spark.createDataFrame(
    [],
    StructType([
        StructField("run_id", LongType(), True),
        StructField("workflow_id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("status", StringType(), True),
        StructField("conclusion", StringType(), True),
        StructField("created_at", StringType(), True),
        StructField("updated_at", StringType(), True),
        StructField("run_number", LongType(), True),
        StructField("repo_id", LongType(), True),
        StructField("repo_full_name", StringType(), True),
        StructField("source_system", StringType(), True),
        StructField("raw_json", StringType(), True),
        StructField("ingestion_timestamp", TimestampType(), True)
    ])
)
df.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.bronze.workflow_runs")

In [0]:
from datetime import datetime, timedelta
import requests
import json

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *

print(f"Analyzing Workflow Runs since {analysis_since.strftime('%Y-%m-%d')}")
spark.sql("TRUNCATE TABLE gitobservatory.bronze.workflow_runs")

config = spark.table("gitobservatory.bronze.repository_config")
repo_metadata = spark.table("gitobservatory.bronze.repository_metadata")
repo_lookup = {}
for row in repo_metadata.collect():
    repo_lookup[row["repo_full_name"]] = row["github_repo_id"]

records = []

for repo in config.collect():
    OWNER = repo["owner"]
    REPOSITORY = repo["repository_name"]
    REPO_FULL_NAME = f"{OWNER}/{REPOSITORY}"
    REPO_ID = repo_lookup.get(REPO_FULL_NAME)
    print(f"\nProcessing {REPO_FULL_NAME}")
    page = 1

    while True:
        url = (
            f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/actions/runs"
            f"?per_page=100&page={page}"
        )
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed for {REPO_FULL_NAME}")
            break
        data = response.json()
        runs = data.get("workflow_runs", [])
        if len(runs) == 0:
            print(f"Finished {REPO_FULL_NAME}")
            break

        print(f"Fetched Page {page}: {len(runs)} workflow runs")
        stop_fetching = False
        for run in runs:
            created_at = datetime.strptime(run["created_at"],"%Y-%m-%dT%H:%M:%SZ")
            if created_at < analysis_since:
                stop_fetching = True
                break

            record = {
                "run_id": run.get("id"),
                "workflow_id": run.get("workflow_id"),
                "name": run.get("name"),
                "status": run.get("status"),
                "conclusion": run.get("conclusion"),
                "created_at": run.get("created_at"),
                "updated_at": run.get("updated_at"),
                "run_number": run.get("run_number"),
                "repo_id": REPO_ID,
                "repo_full_name": REPO_FULL_NAME,
                "source_system": "github",
                "raw_json": json.dumps(run)
            }
            records.append(record)
        if stop_fetching:
            break
        page += 1

schema = StructType([

    StructField("run_id", LongType(), True),
    StructField("workflow_id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("conclusion", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("run_number", LongType(), True),
    StructField("repo_id", LongType(), True),
    StructField("repo_full_name", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)
])

df = spark.createDataFrame(records, schema=schema)
df = df.withColumn("ingestion_timestamp",current_timestamp())
df.write.format("delta").mode("append").saveAsTable("gitobservatory.bronze.workflow_runs")

print(f"\nSuccessfully ingested {len(records)} workflow runs.")

Analyzing Workflow Runs since 2026-06-13


/home/spark-233766c2-48cf-4c12-8882-15/.ipykernel/306/command-7585012712732744-3567340859:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  analysis_since = datetime.utcnow() - timedelta(days=365 * ANALYSIS_YEARS)



Processing dbt-labs/dbt-core
Fetched Page 1: 100 workflow runs
Fetched Page 2: 100 workflow runs
Fetched Page 3: 100 workflow runs
Fetched Page 4: 100 workflow runs
Fetched Page 5: 100 workflow runs
Fetched Page 6: 100 workflow runs
Fetched Page 7: 100 workflow runs
Fetched Page 8: 100 workflow runs
Fetched Page 9: 100 workflow runs
Fetched Page 10: 100 workflow runs
Fetched Page 11: 100 workflow runs
Fetched Page 12: 100 workflow runs
Fetched Page 13: 100 workflow runs
Fetched Page 14: 100 workflow runs
Fetched Page 15: 100 workflow runs
Fetched Page 16: 100 workflow runs
Fetched Page 17: 100 workflow runs
Fetched Page 18: 100 workflow runs
Fetched Page 19: 100 workflow runs
Fetched Page 20: 100 workflow runs
Fetched Page 21: 100 workflow runs
Fetched Page 22: 100 workflow runs
Fetched Page 23: 100 workflow runs
Fetched Page 24: 100 workflow runs
Fetched Page 25: 100 workflow runs
Fetched Page 26: 100 workflow runs
Fetched Page 27: 100 workflow runs
Fetched Page 28: 100 workflow runs